In [14]:
import pandas as pd
import requests
import sqlite3
import logging


pd.set_option('display.max_columns', None)

logging.basicConfig(
    level=logging.INFO,
    filename="logs.log",
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

In [15]:
USERS_URL = "https://jsonplaceholder.typicode.com/users"
POSTS_URL = "https://jsonplaceholder.typicode.com/posts"

In [ ]:
def fetch_data(url):
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        logger.info("Fetched data from %s", url)
        return response.json()
    except requests.RequestException:
        logger.exception("Failed to fetch data from %s", url)
        raise
    except ValueError:
        logger.exception("Invalid JSON received from %s", url)
        
        raise

try:
    users_data = fetch_data(USERS_URL)
    posts_data = fetch_data(POSTS_URL)
except Exception:
    logger.exception("Stopping notebook because source data could not be loaded")
    raise

__main__
__main__


In [18]:
try:
    df_users_raw = pd.DataFrame(users_data)
    df_posts_raw = pd.DataFrame(posts_data)
except ValueError:
    logger.exception("Failed to convert fetched data into DataFrames")
    raise

df_users_raw.head()

,id,name,username,email,address,phone,website,company
0,1,Leanne Graham,Bret,Sincere@april.biz,"{'street': 'Kulas Light', 'suite': 'Apt. 556',...",1-770-736-8031 x56442,hildegard.org,"{'name': 'Romaguera-Crona', 'catchPhrase': 'Mu..."
1,2,Ervin Howell,Antonette,Shanna@melissa.tv,"{'street': 'Victor Plains', 'suite': 'Suite 87...",010-692-6593 x09125,anastasia.net,"{'name': 'Deckow-Crist', 'catchPhrase': 'Proac..."
2,3,Clementine Bauch,Samantha,Nathan@yesenia.net,"{'street': 'Douglas Extension', 'suite': 'Suit...",1-463-123-4447,ramiro.info,"{'name': 'Romaguera-Jacobson', 'catchPhrase': ..."
3,4,Patricia Lebsack,Karianne,Julianne.OConner@kory.org,"{'street': 'Hoeger Mall', 'suite': 'Apt. 692',...",493-170-9623 x156,kale.biz,"{'name': 'Robel-Corkery', 'catchPhrase': 'Mult..."
4,5,Chelsey Dietrich,Kamren,Lucio_Hettinger@annie.ca,"{'street': 'Skiles Walks', 'suite': 'Suite 351...",(254)954-1289,demarco.info,"{'name': 'Keebler LLC', 'catchPhrase': 'User-c..."


In [19]:
# Flatten nested JSON (address.city extraction requirement)
try:
    df_users = pd.json_normalize(users_data)

    required_user_columns = ['id', 'name', 'email', 'address.city']
    missing_user_columns = [column for column in required_user_columns if column not in df_users.columns]
    if missing_user_columns:
        raise KeyError(f"Missing user columns: {missing_user_columns}")

    df_users = df_users[required_user_columns].rename(columns={'address.city': 'city'})
except (KeyError, ValueError):
    logger.exception("Failed to prepare the users DataFrame")
    raise

df_users.head()

,id,name,email,city
0,1,Leanne Graham,Sincere@april.biz,Gwenborough
1,2,Ervin Howell,Shanna@melissa.tv,Wisokyburgh
2,3,Clementine Bauch,Nathan@yesenia.net,McKenziehaven
3,4,Patricia Lebsack,Julianne.OConner@kory.org,South Elvis
4,5,Chelsey Dietrich,Lucio_Hettinger@annie.ca,Roscoeview


In [20]:
try:
    required_post_columns = ['userId', 'title']
    missing_post_columns = [column for column in required_post_columns if column not in df_posts_raw.columns]
    if missing_post_columns:
        raise KeyError(f"Missing post columns: {missing_post_columns}")

    df_posts = df_posts_raw[required_post_columns].rename(columns={'userId': 'id'})
except KeyError:
    logger.exception("Failed to prepare the posts DataFrame")
    raise

df_posts.head()

,id,title
0,1,sunt aut facere repellat provident occaecati e...
1,1,qui est esse
2,1,ea molestias quasi exercitationem repellat qui...
3,1,eum et est occaecati
4,1,nesciunt quas odio


In [21]:
try:
    df_merged = pd.merge(df_users, df_posts, on='id', how='inner')
except ValueError:
    logger.exception("Failed to merge users and posts DataFrames")
    raise

df_merged.head()

,id,name,email,city,title
0,1,Leanne Graham,Sincere@april.biz,Gwenborough,sunt aut facere repellat provident occaecati e...
1,1,Leanne Graham,Sincere@april.biz,Gwenborough,qui est esse
2,1,Leanne Graham,Sincere@april.biz,Gwenborough,ea molestias quasi exercitationem repellat qui...
3,1,Leanne Graham,Sincere@april.biz,Gwenborough,eum et est occaecati
4,1,Leanne Graham,Sincere@april.biz,Gwenborough,nesciunt quas odio


In [22]:
try:
    post_counts = df_posts.groupby('id').size().reset_index(name='post_count')
    df_users = df_users.merge(post_counts, on='id', how='left')

    # Fill users with no posts
    df_users['post_count'] = df_users['post_count'].fillna(0).astype(int)
except KeyError:
    logger.exception("Failed to compute post counts or merge them into users")
    raise

df_users.head()

,id,name,email,city,post_count
0,1,Leanne Graham,Sincere@april.biz,Gwenborough,10
1,2,Ervin Howell,Shanna@melissa.tv,Wisokyburgh,10
2,3,Clementine Bauch,Nathan@yesenia.net,McKenziehaven,10
3,4,Patricia Lebsack,Julianne.OConner@kory.org,South Elvis,10
4,5,Chelsey Dietrich,Lucio_Hettinger@annie.ca,Roscoeview,10


In [28]:
df_merged.head()

,id,name,email,city,title
0,1,Leanne Graham,Sincere@april.biz,Gwenborough,sunt aut facere repellat provident occaecati e...
1,1,Leanne Graham,Sincere@april.biz,Gwenborough,qui est esse
2,1,Leanne Graham,Sincere@april.biz,Gwenborough,ea molestias quasi exercitationem repellat qui...
3,1,Leanne Graham,Sincere@april.biz,Gwenborough,eum et est occaecati
4,1,Leanne Graham,Sincere@april.biz,Gwenborough,nesciunt quas odio


In [29]:
df_users.head()

,id,name,email,city,post_count
0,1,Leanne Graham,sincere@april.biz,Gwenborough,10
1,2,Ervin Howell,shanna@melissa.tv,Wisokyburgh,10
2,3,Clementine Bauch,nathan@yesenia.net,McKenziehaven,10
3,4,Patricia Lebsack,julianne.oconner@kory.org,South Elvis,10
4,5,Chelsey Dietrich,lucio_hettinger@annie.ca,Roscoeview,10


In [ ]:
# cleaning 
try:
    for column in ['name', 'city', 'email']:
        if column not in df_users.columns:
            raise KeyError(f"Missing column in users DataFrame: {column}")

    if 'title' not in df_merged.columns:
        raise KeyError("Missing column in merged DataFrame: title")

    df_users['name'] = df_users['name'].astype(str).str.strip()
    df_users['city'] = df_users['city'].astype(str).str.strip()
    df_users['email'] = df_users['email'].astype(str).str.lower()
    df_merged['title'] = df_merged['title'].astype(str).str.strip()

    # Drop nulls 
    df_users.dropna(inplace=True)
    df_merged.dropna(inplace=True)
except KeyError:
    logger.exception("Cleaning failed because one or more required columns are missing")
    raise

In [24]:
top_users = df_users.sort_values(by='post_count', ascending=False).head(3)

top_users

,id,name,email,city,post_count
0,1,Leanne Graham,Sincere@april.biz,Gwenborough,10
1,2,Ervin Howell,Shanna@melissa.tv,Wisokyburgh,10
2,3,Clementine Bauch,Nathan@yesenia.net,McKenziehaven,10


In [25]:
try:
    df_merged.to_csv("merged_data.csv", index=False)
    logger.info("Saved merged_data.csv successfully")
except OSError:
    logger.exception("Failed to save merged_data.csv")
    raise

In [26]:
try:
    with sqlite3.connect("merged.db") as conn:
        df_merged.to_sql("merged_data", conn, if_exists="replace", index=False)
        df_users.to_sql("users_summary", conn, if_exists="replace", index=False)
    logger.info("Saved to SQLite database successfully")
except sqlite3.Error:
    logger.exception("Failed to save data to SQLite database")
    raise